####Configuration Step1: How to create Databricks Token
- In your Databricks workspace, click your username in the top right corner and select Settings.
- Click Developer.
- Next to Access tokens, click Manage.
- Click Generate new token.

####Configuration Step2: Create the below scope in CLI


In [0]:
#databricks secrets create-scope genaiscope
#databricks secrets put-secret genaiscope databricks_ai_token
DATABRICKS_TOKEN = dbutils.secrets.get(scope="genaiscope", key="databricks_ai_token")
print(DATABRICKS_TOKEN)

####Configuration Step3: Testing the Token & API configurations

- To connect Programmatically with some AI LLM Models, I have to follow few steps:
Identify Databricks Token(Password).
- I need to install a openai client (like chrome browser) to connect with the respective LLM models.
- Instantiate openai client class as an object.
- I use the token identified in the step1 to connect with the required LLM model using the openai client object.
- Collect the API gateway url going to AI Gateway -> click on the respective model databricks-meta-llama-3-1-8b-instruct -> Choose Python tab -> Copy the base url

In [0]:
from openai import OpenAI #Open AI client is a famous client for accessing any AI models
import os
#OpenAI is like a car to reach some destination
#api_key is an id card for authentication everywhere
#base_url is the gateway & road for the destination
#model is the destination
client = OpenAI(api_key=DATABRICKS_TOKEN, base_url="https://1553827837479697.ai-gateway.cloud.databricks.com/mlflow/v1")
response = client.chat.completions.create(
    model="databricks-meta-llama-3-1-8b-instruct",
    messages=[
        {
            "role": "user",
            "content": "Why i have to use openai client to connect with the databricks meta models?"
        }
    ],
    max_tokens=5000
)

print(response.choices[0].message.content)

####Usecase Implementation

![](automated mail generator.jpg)

####Automated Customer Review Response Generator (E-Commerce)
Instead of just classifying a negative review, We can use LLM (NLG) to actually write the email response to the customer.

The Pipeline Concept: Read the negative review from a Delta table, pass it to the LLM with strict instructions, and output a drafted email into a new column.

LLM & GenAI: Using Llama understand the nuanced frustration in the review.

NLG (Natural Language Generation): The model synthesizes a polite, grammatically perfect, and empathetic email.

Grounding: Pass the company's official return policy into the system prompt. The AI is grounded in this specific document so it doesn't make up rules.

Guardrailing: Add a strict rule to the prompt: "Never promise a refund or discount. Only offer to connect them to the support team."

Hallucination (Anti) Mitigation: Set temperature=0.1, so the AI doesn't invent fake product features or invent a fake customer service rep name.

####Step 1: Create the Company Policy PDF

In [0]:
from fpdf import FPDF

# 1. Create the PDF with the Grounding Context
pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)

policies = """Company Policies:
- We only accept returns within 30 days of purchase.
- We do NOT give cash refunds. We only offer store credit or exact item replacements.
- Standard shipping delays are currently expected due to weather."""

pdf.multi_cell(0, 10, txt=policies)
pdf.output("/tmp/company_policies.pdf")

print("PDF successfully generated at /tmp/company_policies.pdf")

####Step 2: Create the Raw Review Data (Bronze Table)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS lakehousecat;
CREATE SCHEMA IF NOT EXISTS lakehousecat.default;
DROP TABLE IF EXISTS lakehousecat.default.customer_reviews;

CREATE TABLE IF NOT EXISTS lakehousecat.default.customer_reviews (
  review_id STRING,
  customer_name STRING,
  date_of_purchase DATE,
  city STRING,
  review_text STRING);

INSERT INTO lakehousecat.default.customer_reviews VALUES
("REV-001", "Alice", DATE '2026-02-15', 'NYC', "The shoe are beautiful, but they ripped after two days of wearing them! I want my money back."),
("REV-002", "Bob", DATE '2026-01-10', 'NY', "The Laptop Shipping took 3 weeks. Absolutely unacceptable."),
("REV-003", "Charlie", DATE '2026-03-01', 'CA', "Perfect fit! Will definitely be buying from you guys again."),
("REV-004", "Diana", DATE '2026-02-28', 'CAL', "I received the wrong color. I ordered black but got blue. How do I fix this?");

####Step 3: Engineer the GenAI System Prompt
This is the most critical part of the pipeline. We use the system prompt to apply our AI techniques:

**Grounding:** We inject the company's actual return policy into the prompt so the AI doesn't invent rules.

**Guardrailing:** We explicitly forbid the AI from offering cash refunds or discounts.

**Hallucination Mitigation:** We tell it exactly what to say if it doesn't know the answer, preventing it from making up fake support phone numbers.

**NLG:** We constrain the length and tone of the generated text.

####Step 4: Define the PySpark UDF with the AI Gateway

####Step 5: Run the Pipeline and Generate Responses, store to Silver

In [0]:
import os
import PyPDF2
from datetime import datetime
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
from openai import OpenAI

# Fetch your token securely
DATABRICKS_TOKEN = dbutils.secrets.get(scope="izgenaiscope2", key="databricks_token")

# Get today's date dynamically for the AI to calculate the 30-day window
today_date = datetime.today().strftime('%Y-%m-%d')

# --- Extract Grounding Context from PDF ---
def get_pdf_text(filepath):
    text = ""
    with open(filepath, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

# Read the PDF generated in Step 1
pdf_grounding_context = get_pdf_text("/tmp/company_policies.pdf")

# Define the prompt dynamically using the PDF text and today's date
genai_system_prompt = f"""
You are an empathetic, professional customer support agent for an E-Commerce shoe company.
Read the customer's review and write a direct email response to them. Ensure you address them by their name. Apply the guardrails to calculate the number of days of purchase and display the total days of purchase till today's date. In the mail signature add 'inceptez technologies' as the company name.

CURRENT DATE: {today_date} (Use this to calculate if a return is valid).

GROUNDING CONTEXT (Company Policies extracted from document):
{pdf_grounding_context}

GUARDRAILS (Strict Rules):
1. Never promise a cash refund under any circumstances.
2. Never offer a discount code or coupon.
3. Keep the response under 4 sentences.
4. Check for the days of current date minus purchase date, if it is greater than 30 days then respond as the return window is closed, else allow the return to prgress following the grounding context.

HALLUCINATION MITIGATION:
If the user asks for something not covered in the policies above, do not invent a solution. Simply apologize and state: "I am escalating this to our senior support team who will contact you within 24 hours." Do not make up fake phone numbers, links, or names.
"""
#4. If they are asking for a return/refund and their purchase date is MORE than 30 days before the CURRENT DATE, politely decline the return based on policy.
def generate_review_response_udf(customer_name, review_text, date_of_purchase):
    # Initialize client inside the function for Spark worker node compatibility
    client = OpenAI(
        api_key=DATABRICKS_TOKEN,
        base_url="https://3631047205072094.ai-gateway.cloud.databricks.com/mlflow/v1"
    )
    
    try:
        response = client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                {"role": "system", "content": genai_system_prompt},
                # Pass name, review, AND purchase date to the AI
                {"role": "user", "content": f"Customer Name: {customer_name}\nPurchase Date: {date_of_purchase}\nCustomer Review: {review_text}"}
            ],
            max_tokens=1500,    
            temperature=1    
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error Generating Response: {str(e)}"

# Register the Python function as a Spark UDF (Now accepts three string columns)
spark_ai_responder = udf(generate_review_response_udf, StringType())

# Read the raw reviews 
df_raw_reviews = spark.read.table("lakehousecat.default.customer_reviews")

# Apply the GenAI UDF to generate the email responses, casting date to string
df_responded = df_raw_reviews.withColumn(
    "ai_generated_email", 
    spark_ai_responder(col("customer_name"), col("review_text"), col("date_of_purchase").cast(StringType())))

# Display the output for quick verification
display(df_responded)

# --- Store the final data into a new Delta Table ---
(df_responded.write
    .format("delta")
    .mode("overwrite") # Use "append" if you want to add to existing data instead
    .saveAsTable("lakehousecat.default.customer_reviews_responded"))

print("Data successfully processed and saved to lakehousecat.default.customer_reviews_responded")